# Level 3: Multi-Qubit Systems and the Magic of Entanglement
## Qiskit Edition

In this notebook, you will:

1. Work with **multi-qubit systems**
2. Understand **entanglement** — Einstein's "spooky action at a distance"
3. Build your first **Bell State** using the CNOT gate
4. Verify entanglement through **correlated measurements**

---
## 1. Multi-Qubit Systems

### Combining Qubits

With 1 qubit, we have 2 possible states: |0> and |1>.
With 2 qubits, we have 4 possible states: |00>, |01>, |10>, |11>.
With n qubits, we have 2^n possible states!

This exponential growth is what makes quantum computing powerful.

### What Is Entanglement?

**Entanglement** is a uniquely quantum phenomenon where two qubits become correlated in such a way that:
- Measuring one qubit **instantly** determines the state of the other
- This correlation holds **regardless of distance**
- There is **no classical equivalent**

Einstein famously called this "spooky action at a distance" because he found it disturbing. But experiments have confirmed it's real!

### The CNOT Gate

The **Controlled-NOT (CNOT)** gate is a two-qubit gate:
- It has a **control** qubit and a **target** qubit
- If the control is |1>, it flips the target
- If the control is |0>, nothing happens

| Control | Target | Output Control | Output Target |
|---------|--------|:--------------:|:-------------:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 1 | 1 |
| 1 | 1 | 1 | 0 |

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt

simulator = AerSimulator()
print("Ready!")

---
## 2. The CNOT Gate in Action

In [ ]:
# CNOT with control=|0>: target stays the same
qc1 = QuantumCircuit(2, 2)
# Both qubits start as |0>
qc1.cx(0, 1)  # CNOT: control=qubit 0, target=qubit 1
qc1.measure([0, 1], [0, 1])

result1 = simulator.run(qc1, shots=1000).result().get_counts()
print(f"CNOT with control=|0>: {result1}")
print("Nothing changes because control is 0!")

qc1.draw('mpl')

In [ ]:
# CNOT with control=|1>: target gets flipped
qc2 = QuantumCircuit(2, 2)
qc2.x(0)       # Set control to |1>
qc2.cx(0, 1)   # CNOT: flips target because control is |1>
qc2.measure([0, 1], [0, 1])

result2 = simulator.run(qc2, shots=1000).result().get_counts()
print(f"CNOT with control=|1>: {result2}")
print("Both qubits are now |1>!")

qc2.draw('mpl')

---
## 3. Creating a Bell State (Entanglement!)

The **Bell State** is the simplest entangled state. The recipe is:
1. Apply **H** to the first qubit (creates superposition)
2. Apply **CNOT** with qubit 0 as control and qubit 1 as target

This creates the state: (1/sqrt(2))(|00> + |11>)

In this state:
- If you measure qubit 0 and get 0, qubit 1 is **guaranteed** to be 0
- If you measure qubit 0 and get 1, qubit 1 is **guaranteed** to be 1
- You'll never see |01> or |10>!

In [ ]:
# Create the Bell State
bell = QuantumCircuit(2, 2)

# Step 1: Hadamard on qubit 0
bell.h(0)

# Step 2: CNOT with qubit 0 as control, qubit 1 as target
bell.cx(0, 1)

# Let's look at the state before measuring
state_bell = Statevector.from_instruction(bell)
print("Bell State:")
print(state_bell)
print("\nNotice: only |00> and |11> have non-zero amplitudes!")

bell.draw('mpl')

In [ ]:
# Now measure and run many times
bell_measure = QuantumCircuit(2, 2)
bell_measure.h(0)
bell_measure.cx(0, 1)
bell_measure.measure([0, 1], [0, 1])

result_bell = simulator.run(bell_measure, shots=10000).result().get_counts()
print(f"Bell State measurements: {result_bell}")
print("\nOnly '00' and '11' appear — the qubits are perfectly correlated!")
print("This is ENTANGLEMENT. Measuring one determines the other.")

plot_histogram(result_bell)

---
## 4. All Four Bell States

There are actually four Bell States, each representing a different kind of perfect correlation:

| Bell State | Circuit | State |
|------------|---------|-------|
| Phi+ | H, CNOT | (1/sqrt(2))(\|00> + \|11>) |
| Phi- | H, CNOT, Z | (1/sqrt(2))(\|00> - \|11>) |
| Psi+ | H, CNOT, X | (1/sqrt(2))(\|01> + \|10>) |
| Psi- | H, CNOT, Z, X | (1/sqrt(2))(\|01> - \|10>) |

In [ ]:
# Create all four Bell States
def create_bell_state(variant):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    
    if variant == "phi-":
        qc.z(0)
    elif variant == "psi+":
        qc.x(0)
    elif variant == "psi-":
        qc.z(0)
        qc.x(0)
    
    qc.measure([0, 1], [0, 1])
    return qc

bells = {"Phi+": "phi+", "Phi-": "phi-", "Psi+": "psi+", "Psi-": "psi-"}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, variant) in zip(axes, bells.items()):
    qc = create_bell_state(variant)
    counts = simulator.run(qc, shots=10000).result().get_counts()
    plot_histogram(counts, ax=ax, title=f"Bell State {name}")

plt.tight_layout()
plt.show()

---
## 5. Entanglement Is Not Cloning

A common misconception: entanglement does NOT copy the state of one qubit to another.

The **No-Cloning Theorem** states that you cannot create an exact copy of an unknown quantum state. This is fundamental to quantum mechanics and is what makes quantum cryptography secure.

Let's demonstrate: if we put qubit 0 in superposition and CNOT it with qubit 1, qubit 1 does NOT become an independent copy.

In [ ]:
# The qubits are correlated, not independent copies
# If they were independent copies, we'd see all four outcomes: 00, 01, 10, 11
# But we only see 00 and 11 — they're entangled!

print("If qubits were independent copies (50/50 each):")
print("  Expected: 25% |00>, 25% |01>, 25% |10>, 25% |11>")
print(f"\nActual Bell State results: {result_bell}")
print("  We see: ~50% |00>, ~50% |11>, 0% |01>, 0% |10>")
print("\nThis perfect correlation is the signature of entanglement!")

---
## 6. Three-Qubit Entanglement: GHZ State

We can entangle more than two qubits! The **GHZ state** (Greenberger-Horne-Zeilinger) entangles three qubits:

(1/sqrt(2))(|000> + |111>)

In [ ]:
# Create a GHZ state with 3 qubits
ghz = QuantumCircuit(3, 3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(0, 2)
ghz.measure([0, 1, 2], [0, 1, 2])

print("GHZ Circuit:")
print(ghz)

result_ghz = simulator.run(ghz, shots=10000).result().get_counts()
print(f"\nGHZ State results: {result_ghz}")
print("Only |000> and |111> — all three qubits are entangled!")

plot_histogram(result_ghz)

---
## 7. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Multi-qubit** | n qubits = 2^n possible states |
| **CNOT** | Flips target if control is \|1> |
| **Bell State** | Simplest entangled state of 2 qubits |
| **Entanglement** | Measuring one qubit determines the other |
| **No-Cloning** | Cannot copy an unknown quantum state |
| **GHZ State** | Entanglement extended to 3+ qubits |

---
## 8. Challenges

1. **4-Qubit GHZ**: Create a GHZ state with 4 qubits. What outcomes do you see?
2. **Anti-correlated Bell**: Create a Bell state where the qubits are anti-correlated (only |01> and |10> appear).
3. **Swap Test**: Build a circuit that uses CNOT gates to swap the states of two qubits.

In [ ]:
# Your challenge code here!

---
**Next up: [Level 4 — Quantum Protocols: Teleportation & Superdense Coding](../Level_04_Quantum_Protocols/Level_04_Qiskit.ipynb)**